In [4]:
# Creating a model
from cobra import Model, Reaction, Metabolite

model = Model("toy_gut_bug")
glc_e = Metabolite("glc_D_e", name="D-glucose", compartment="e")
glc_c = Metabolite("glc_D_c", name="D-glucose", compartment="c")
pyr_c = Metabolite("pyr_c", name="pyruvate", compartment="c")
but_c = Metabolite("but_c", name="butyrate", compartment="c")
but_e = Metabolite("but_e", name="butyrate", compartment="e")
bio_c = Metabolite("biomass_c", name="biomass", compartment="c")

def rxn(rid, stoich, lb=0, ub=1000, name=""):
    r = Reaction(rid, name=name, lower_bound=lb, upper_bound=ub)
    r.add_metabolites(stoich)
    return r

model.add_reactions([
    rxn("EX_glc_D_e", {glc_e: -1}, lb=-10),           
    rxn("GLCt",   {glc_e: -1, glc_c: 1}, name="glucose transport"),
    rxn("GLYC",   {glc_c: -1, pyr_c: 2}, name="glycolysis (lumped)"),
    rxn("BUTSYN", {pyr_c: -2, but_c: 1}, name="butyrate synthesis (lumped)"),
    rxn("BUTt",   {but_c: -1, but_e: 1}, name="butyrate export"),
    rxn("EX_but_e", {but_e: -1}),
    rxn("BIOMASS", {pyr_c: -5, bio_c: 1}, name="biomass"),
    rxn("EX_biomass", {bio_c: -1}),
])
model.objective = "BIOMASS"
print(len(model.reactions), "reactions,", len(model.metabolites), "metabolites")

8 reactions, 6 metabolites


In [5]:
# Looking at the S matrix. Rows are the metabolites, columns are the reactions
from cobra.util.array import create_stoichiometric_matrix
create_stoichiometric_matrix(model, array_type="DataFrame")

,EX_glc_D_e,GLCt,GLYC,BUTSYN,BUTt,EX_but_e,BIOMASS,EX_biomass
glc_D_e,-1.0,-1.0,0.0,0.0,0.0,0.0,0.0,0.0
glc_D_c,0.0,1.0,-1.0,0.0,0.0,0.0,0.0,0.0
pyr_c,0.0,0.0,2.0,-2.0,0.0,0.0,-5.0,0.0
but_c,0.0,0.0,0.0,1.0,-1.0,0.0,0.0,0.0
but_e,0.0,0.0,0.0,0.0,1.0,-1.0,0.0,0.0
biomass_c,0.0,0.0,0.0,0.0,0.0,0.0,1.0,-1.0


In [6]:
sol = model.optimize()
print(sol.status, "growth =", sol.objective_value)
sol.fluxes

optimal growth = 4.0


EX_glc_D_e   -10.0
GLCt          10.0
GLYC          10.0
BUTSYN         0.0
BUTt           0.0
EX_but_e       0.0
BIOMASS        4.0
EX_biomass     4.0
Name: fluxes, dtype: float64

In [14]:
# Inside a "with model:" block every change is undone when the block ends.
with model:
    model.objective = "EX_but_e"
    print("Maximum butyrate:", model.slim_optimize())
    m = model.optimize()
    print(m.fluxes)

with model:
    model.reactions.EX_glc_D_e.lower_bound = 0   # remove glucose from diet
    print("Growth without glucose:", model.slim_optimize())

print("Bound restored to:", model.reactions.EX_glc_D_e.lower_bound)

Maximum butyrate: 10.0
EX_glc_D_e   -10.0
GLCt          10.0
GLYC          10.0
BUTSYN        10.0
BUTt          10.0
EX_but_e      10.0
BIOMASS        0.0
EX_biomass     0.0
Name: fluxes, dtype: float64
Growth without glucose: 0.0
Bound restored to: -10


In [9]:
import lbpscreen as lbp
from pathlib import Path

MODEL_DIR = Path(r"C:\Users\IPRDC\lbp-screen\data\processed\core_models")
FILE_MAP = {
    "E_faecalis_MMH594":    "Enterococcus_faecalis_MMH594_AGORA2_v2.01.xml",
    "E_lenta_DSM2243":      "Eggerthella_lenta_DSM_2243_AGORA2_v2.01.xml",
    "E_coli_Nissle1917":    "Escherichia_coli_Nissle_1917_AGORA2_v2.01.xml",
    "A_muciniphila_BAA835": "Akkermansia_muciniphila_ATCC_BAA_835_AGORA2_v2.01.xml",
}
models = lbp.load_models(MODEL_DIR, FILE_MAP)
summary = lbp.model_summary(models)
summary["growth_default_bounds"] = [round(m.slim_optimize(), 3) for m in models.values()]
summary

,reactions,metabolites,genes,exchanges,objective,format,growth_default_bounds
strain,,,,,,,
E_faecalis_MMH594,1558,1484,1140,173,bio1,sbml,0.0
E_lenta_DSM2243,1035,968,690,121,biomass467,sbml,0.0
E_coli_Nissle1917,3574,3115,1457,569,bio1,sbml,0.0
A_muciniphila_BAA835,1215,1109,587,118,biomass345,sbml,0.0


In [10]:
# What the E. coli Nissle model consumes and secretes?
models["E_coli_Nissle1917"].summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
Metabolite,Reaction,Flux,C-Number,C-Flux
